<a class="btn btn-primary btn-sm" href="https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/getting_the_data.ipynb" download>&#8681; Download this notebook (.ipynb)</a>

*Optional reference — we won't work through this in class.*

The CSV files used in Lab 0c didn't fall from the sky: they were downloaded from three public archives. This notebook shows exactly how, partly so the numbers are reproducible, and partly because **getting data is a real skill** you'll need for your final project.

The cells here are not executed when this page is built (the downloads are slow, and the class shouldn't depend on three servers being up). Download the notebook and run it yourself if you want the data fresh.

| what | source | note |
|---|---|---|
| station observations | NOAA NCEI, GHCN-Daily | one thermometer, Central Park |
| reanalysis | ERA5 (Copernicus/ECMWF), via Open-Meteo | model + observations, on a grid |
| climate model | CMIP6 MPI-ESM1-2-HR, via the Pangeo cloud archive | free-running simulation |

All three cover **1995–2014** — the last 20 years of the CMIP6 *historical* experiment, so they overlap.

In [ ]:
import io, json, urllib.request
import pandas as pd

LAT, LON = 40.78, -73.97          # Central Park
Y0, Y1 = "1995-01-01", "2014-12-31"

## 1. Station observations — NOAA GHCN-Daily

The Global Historical Climatology Network holds daily records from tens of thousands of stations. Central Park is `USW00094728`. NOAA's Access Data Service will hand you a CSV straight from a URL.

Note we ask for `TMAX` and `TMIN` and average them: daily *mean* temperature (`TAVG`) is missing for many station-years, so `(TMAX + TMIN)/2` is the standard workaround. That is a modelling choice, and it is **not** identical to a true daily mean: checked against ERA5 (which provides both), the midpoint runs about **+0.2 °C warm** here. ERA5 and CMIP6 below give true 24-hour means, so the three series are not quite like-for-like.

In [ ]:
url = ("https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries"
       f"&stations=USW00094728&startDate={Y0}&endDate={Y1}&dataTypes=TMAX,TMIN"
       "&units=metric&format=csv")

obs = pd.read_csv(io.StringIO(urllib.request.urlopen(url, timeout=180).read().decode()))
obs["DATE"] = pd.to_datetime(obs["DATE"])
obs["t_obs"] = (obs["TMAX"] + obs["TMIN"]) / 2
obs = obs.set_index("DATE")[["t_obs"]]
obs.head()

## 2. Reanalysis — ERA5

ERA5 is a weather model run over the whole satellite/observation record, continuously nudged toward observations. The full archive is enormous; [Open-Meteo](https://open-meteo.com/) offers a friendly API that extracts a single point for you.

**Watch the fine print:** you asked for Central Park, but you get the *nearest grid point* — around 40.81 °N, 74.02 °W, and it represents an average over roughly 30 km, not a park.

In [ ]:
url = (f"https://archive-api.open-meteo.com/v1/era5?latitude={LAT}&longitude={LON}"
       f"&start_date={Y0}&end_date={Y1}&daily=temperature_2m_mean"
       "&timezone=America%2FNew_York")

j = json.load(urllib.request.urlopen(url, timeout=300))
era = pd.DataFrame({"DATE": pd.to_datetime(j["daily"]["time"]),
                    "t_era5": j["daily"]["temperature_2m_mean"]}).set_index("DATE")

print("grid point actually returned:", j["latitude"], j["longitude"])
era.head()

In [ ]:
obs.join(era, how="outer").to_csv("centralpark_obs_era5_1995-2014.csv")

## 3. A climate model — CMIP6 from the cloud

This is the one we'll use properly later in the course. Every CMIP6 simulation is catalogued in one big CSV; you filter it to the run you want, then open the data straight from cloud storage — **without downloading the whole thing** (`xarray` fetches only the chunks you touch).

The search terms are the CMIP vocabulary you'll learn later: `source_id` (which model), `experiment_id` (which experiment), `member_id` (which ensemble member), `table_id` (which output frequency), `variable_id` (which variable — `tas` is near-surface air temperature).

In [ ]:
import xarray as xr

cat = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")
print(len(cat), "simulations in the catalog")

sel_all = cat[(cat.experiment_id == "historical") & (cat.member_id == "r1i1p1f1")
               & (cat.table_id == "day") & (cat.variable_id == "tas")]

sel = cat[(cat.source_id == "MPI-ESM1-2-HR") & (cat.experiment_id == "historical")
          & (cat.member_id == "r1i1p1f1") & (cat.table_id == "day")
          & (cat.variable_id == "tas")]
sel[["source_id", "experiment_id", "member_id", "table_id", "variable_id", "zstore"]]

In [ ]:
ds = xr.open_zarr(sel.zstore.iloc[0], consolidated=True, storage_options={"token": "anon"})
ds

Four things bite everyone the first time:

1. **Longitude runs 0–360**, not −180–180. Central Park's −73.97 becomes `286.03` (`LON % 360`).
2. **Temperature is in kelvin.** Subtract 273.15.
3. **The nearest grid cell is ~100 km wide** — here it lands at about 40.68 °N, 74.06 °W, partly over water. It is *not* Central Park.
4. **Model calendars are their own thing** (`cftime` objects, sometimes 360-day years), so converting to ordinary dates takes an extra step.

In [ ]:
pt = (ds.tas.sel(lat=LAT, lon=LON % 360, method="nearest")
            .sel(time=slice("1995", "2014")) - 273.15).load()

print(f"grid cell used: {float(pt.lat):.2f} N, {float(pt.lon) - 360:.2f} E")

s = pt.to_series()
s.index = pd.to_datetime(s.index.astype(str))     # cftime -> ordinary dates
s.rename("t_cmip6").to_csv("centralpark_cmip6_MPI-ESM1-2-HR_1995-2014.csv")
s.head()

## 4. The same thing, for eight models

One model is one opinion. The lab also uses an **ensemble**: the identical extraction, looped over eight modelling centres. Each model has its own grid, so "the cell nearest Central Park" is a different patch of the planet in each case — for the coarsest model it lands over the ocean.

In [ ]:
MODELS = ["MPI-ESM1-2-HR", "MPI-ESM1-2-LR", "CanESM5", "MIROC6",
          "IPSL-CM6A-LR", "ACCESS-CM2", "NorESM2-LM", "GFDL-ESM4"]

out = {}
for m in MODELS:
    try:
        z = sel_all[sel_all.source_id == m].zstore.iloc[0]
        ds_m = xr.open_zarr(z, consolidated=True, storage_options={"token": "anon"})
        pt_m = (ds_m.tas.sel(lat=LAT, lon=LON % 360, method="nearest")
                       .sel(time=slice("1995", "2014")) - 273.15).load()
        s_m = pt_m.to_series()
        s_m.index = pd.to_datetime(s_m.index.astype(str)).normalize()
        out[m] = s_m[~s_m.index.duplicated()]
        print(f"{m:16s} cell {float(pt_m.lat):6.2f}N {float(pt_m.lon)-360:7.2f}E   mean {s_m.mean():5.2f} C")
    except Exception as exc:
        print(f"{m:16s} FAILED ({type(exc).__name__})")

models = pd.DataFrame(out)
models.index.name = "time"
models.to_csv("centralpark_cmip6_multimodel_1995-2014.csv")

## Credit where due

- **GHCN-Daily** — NOAA National Centers for Environmental Information (US government work, public domain).
- **ERA5** — Copernicus Climate Change Service / ECMWF; served here by Open-Meteo.
- **MPI-ESM1-2-HR** — Max Planck Institute for Meteorology, distributed through CMIP6 (CC BY 4.0). Cloud copy hosted by [Pangeo](https://pangeo.io/).

Citing your data sources is not a formality — it's how someone else (including future you) can check what you did.

**Want more of this?** Working with real earth-science datasets — xarray, cloud archives, plotting, analysis — is the subject of the companion course site: <https://earth-ds-ml.github.io/summer_2026/intro.html>